In [0]:
%sql
SHOW TABLES IN formula1_catalog.bronze

In [0]:
%run ../00-common/01.environment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"


In [0]:

bronze_table

In [0]:
%sql
describe history formula1_catalog.bronze.results

In [0]:
results_df = (
    spark.table(bronze_table)
    .select(
        "season",
        "round",
        "constructorId",
        "driverId",
        "date",
        "raceName",
        "grid",
        "laps",
        "number",
        "points",
        "position",
        "positionText",
        "status",
        "ingestion_timestamp",
        "source_file",
    )
    .withColumnsRenamed(
        {
            "driverId": "driver_id",
            "date": "race_date",
            "constructorId": "constructor_id",
            "raceName": "race_name",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "points": "race_points",
            "position": "funal_position",
            "positionText": "final_position_text",
        }
    )
    .filter(
        F.col("season").isNotNull()
        & F.col("round").isNotNull()
        & F.col("constructor_id").isNotNull()
        & F.col("driver_id").isNotNull()
    )
    .dropDuplicates(["driver_id", "season", "round", "constructor_id"])
    .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(results_df)

In [0]:
(
    results_df
        .write
        .format("delta")
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1_catalog.silver.results